In [0]:
from pyspark.sql.functions import current_timestamp, col

SOURCE_PATH = "/Volumes/workspace/oil_gas_bronze/landing/telemetry"
CHECKPOINT_PATH = "/Volumes/workspace/oil_gas_bronze/landing/_checkpoints/telemetry_raw"
TARGET_TABLE = "workspace.oil_gas_bronze.telemetry_raw"

raw_schema = "well_id STRING, sensor_timestamp STRING, wellhead_pressure_raw STRING, flow_bpd_raw STRING"

df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "json")
      .schema(raw_schema)
      .load(SOURCE_PATH)
      .select("*", col("_metadata.file_path").alias("source_file"))
      .withColumn("ingested_at", current_timestamp()))

In [0]:
query = (df.writeStream
         .option("checkpointLocation", CHECKPOINT_PATH)
         .trigger(availableNow=True)
         .toTable(TARGET_TABLE))

query.awaitTermination()

In [0]:
%sql
SELECT COUNT(*) AS rows_total,
       COUNT(DISTINCT well_id, sensor_timestamp) AS distinct_readings,
       COUNT(DISTINCT source_file) AS files
FROM workspace.oil_gas_bronze.telemetry_raw;